# YOLO26 video experiment

Run pretrained YOLO26 on the supplied video. The progress bar updates once per processed frame.

In [7]:
from pathlib import Path
import csv
import cv2
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from ultralytics import YOLO

In [8]:
VIDEO_PATH = Path('../data/Neural_net_ninja_video.mp4')
MODEL_NAME = 'yolo26n.pt'
OUTPUT_DIR = Path('runs/yolo26-notebook')
CONFIDENCE = 0.25
IMAGE_SIZE = 640
MAX_FRAMES = None  # Set to a small integer for a quick test.

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f'Video does not exist: {VIDEO_PATH.resolve()}')
if not 0 < CONFIDENCE <= 1:
    raise ValueError('CONFIDENCE must be greater than 0 and at most 1')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / f'{VIDEO_PATH.stem}_annotated.mp4'
OUTPUT_CSV = OUTPUT_DIR / f'{VIDEO_PATH.stem}_detections.csv'
print(f'Input: {VIDEO_PATH}')
print(f'Model: {MODEL_NAME}')
print(f'Output directory: {OUTPUT_DIR}')

Input: ..\data\Neural_net_ninja_video.mp4
Model: yolo26n.pt
Output directory: runs\yolo26-notebook


In [9]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f'Could not open input video: {VIDEO_PATH}')

width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
if MAX_FRAMES is not None:
    total_frames = min(total_frames, MAX_FRAMES)

writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
if not writer.isOpened():
    capture.release()
    raise RuntimeError(f'Could not create output video: {OUTPUT_VIDEO}')

model = YOLO(MODEL_NAME)
frame_number = 0
detection_count = 0
try:
    with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(['frame', 'class_id', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2'])
        with tqdm(total=total_frames, desc='Processing video', unit='frame') as progress:
            while MAX_FRAMES is None or frame_number < MAX_FRAMES:
                ok, frame = capture.read()
                if not ok:
                    break
                result = model.predict(source=frame, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
                writer.write(result.plot())
                if result.boxes is not None:
                    for box in result.boxes:
                        class_id = int(box.cls[0])
                        confidence = float(box.conf[0])
                        x1, y1, x2, y2 = (float(value) for value in box.xyxy[0])
                        csv_writer.writerow([frame_number, class_id, result.names[class_id], f'{confidence:.4f}', f'{x1:.2f}', f'{y1:.2f}', f'{x2:.2f}', f'{y2:.2f}'])
                        detection_count += 1
                frame_number += 1
                progress.update(1)
finally:
    capture.release()
    writer.release()

print(f'Processed frames: {frame_number}')
print(f'Detections: {detection_count}')
print(f'Annotated video: {OUTPUT_VIDEO}')
print(f'Detections CSV: {OUTPUT_CSV}')

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# Optional: display the first annotated frame.
preview = cv2.VideoCapture(str(OUTPUT_VIDEO))
ok, frame = preview.read()
preview.release()
if ok:
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()